# Perbandingan 1D-CNN vs RNN vs LSTM vs GRU vs DNN vs ARIMA untuk Peramalan Revenue Mingguan (Dataset M5)

**Tujuan notebook ini:** membandingkan performa model **1D-CNN**, **RNN**, **LSTM**, **GRU**, **DNN**, dan **ARIMA** dalam meramalkan
revenue mingguan per kombinasi *state* x *kategori* pada dataset M5 Forecasting Accuracy,
menggunakan skema evaluasi **walk-forward** yang **identik** untuk keenam model agar perbandingannya adil (apple-to-apple),
dan dievaluasi dengan **RMSSE per deret** serta **WRMSSE (Weighted RMSSE)** ala kompetisi M5.


## 1. Import Library

In [1]:
import os
import gc
import itertools
import warnings

from IPython.display import display  # supaya display() tetap jalan walau di luar Colab

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

import resource
import ctypes

def print_ram(label=""):
    """Cetak pemakaian RAM proses saat ini (RSS, dalam MB) supaya kita bisa memantau
    apakah memori terus naik (leak) selama loop panjang berjalan."""
    rss_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss  # KB di Linux (termasuk Colab)
    print(f"[RAM] {label} peak RSS ~ {rss_kb / 1024:.1f} MB")

def free_memory(*objs):
    """Bersihkan memori seagresif mungkin setelah sebuah model Keras selesai dipakai.

    PENTING: memanggil del + K.clear_session() + gc.collect() saja SERINGKALI TIDAK CUKUP
    di Colab/Linux, karena allocator memori C (glibc malloc) tidak selalu mengembalikan
    memori yang sudah dibebaskan kembali ke sistem operasi -- sehingga RSS proses terus
    terlihat naik walau objek Python-nya sendiri sudah dihapus. Ini yang menyebabkan sesi
    Colab crash 'kehabisan RAM' walau notebook ini sudah memanggil clear_session()/gc.collect()
    di setiap iterasi. Fungsi ini menambahkan langkah `malloc_trim(0)` di akhir untuk memaksa
    memori yang sudah bebas benar-benar dikembalikan ke OS.
    """
    for o in objs:
        try:
            del o
        except Exception:
            pass
    try:
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc.collect()
    gc.collect()  # dipanggil 2x: generasi lama kadang butuh siklus collect kedua
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass  # bukan Linux / libc tidak tersedia -> lewati saja, tidak fatal

# --- Perbaikan anti-crash RAM: batasi TensorFlow agar tidak langsung menyita seluruh memori GPU ---
import tensorflow as tf
for _gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception:
        pass

# --- Perbaikan anti-crash RAM (tambahan): banyak model kecil dibangun & dibuang berulang kali
# (ratusan-ribuan kali) di notebook ini. TensorFlow secara default meng-cache 'trace' fungsi
# (tf.function) untuk tiap model baru, dan cache ini bisa menumpuk terus walau modelnya sudah
# dihapus. Menjalankan secara eager (tanpa tracing/XLA) untuk model-model kecil ini jauh lebih
# aman untuk RAM, dengan trade-off kecepatan yang bisa diabaikan karena modelnya kecil.
tf.config.run_functions_eagerly(True)
try:
    tf.config.optimizer.set_jit(False)
except Exception:
    pass


## 2. Unduh Dataset M5

Dataset diunduh dari Kaggle (`aryayadav0513/m5-forecasting-accuracy`) menggunakan `kagglehub`.
File yang dipakai: `sales_train_validation.csv`, `calendar.csv`, `sell_prices.csv`.

In [2]:
import kagglehub

path = kagglehub.dataset_download("aryayadav0513/m5-forecasting-accuracy")
dataset_path = os.path.join(path, "m5-forecasting-accuracy")
print("Path dataset:", dataset_path)
print("Isi folder :", os.listdir(dataset_path))


Using Colab cache for faster access to the 'm5-forecasting-accuracy' dataset.
Path dataset: /kaggle/input/m5-forecasting-accuracy/m5-forecasting-accuracy
Isi folder : ['calendar.csv', 'sample_submission.csv', 'sell_prices.csv', 'sales_train_validation.csv', 'sales_train_evaluation.csv']


## 3. Muat & Praproses Data

Langkah-langkah:
1. Muat `sales_train_validation.csv`, `calendar.csv`, `sell_prices.csv`.
2. Filter hanya kombinasi *state* (`CA`, `TX`, `WI`) dan *kategori* (`HOBBIES`, `HOUSEHOLD`, `FOODS`) yang dipakai di eksperimen ini.
3. Ambil sampel 50% item dan separuh kolom hari (`d_*`) pertama agar ukuran data lebih ringan untuk eksperimen.
4. Ubah ke format *long* (`melt`), lalu gabungkan dengan `calendar` (untuk `wm_yr_wk`) dan `sell_prices` (untuk harga jual).


In [3]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

print("sales   :", sales.shape)
print("calendar:", calendar.shape)
print("prices  :", prices.shape)


sales   : (30490, 1919)
calendar: (1969, 14)
prices  : (6841121, 4)


In [4]:
STATES = ['CA', 'TX', 'WI']
CATEGORIES = ['HOBBIES', 'HOUSEHOLD', 'FOODS']

# Filter state & kategori yang relevan, lalu ambil sampel 50% item untuk mempercepat eksperimen
sales = sales[
    sales['state_id'].isin(STATES) &
    sales['cat_id'].isin(CATEGORIES)
].sample(frac=0.5, random_state=SEED)

# Ambil separuh kolom hari pertama (mengurangi beban komputasi)
day_cols = [c for c in sales.columns if c.startswith('d_')]
half_days = day_cols[: len(day_cols) // 2]
sales = sales[['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'] + half_days]

print("Bentuk data setelah filter & sampling:", sales.shape)

# --- Perbaikan anti-crash RAM: turunkan tipe data ---
# Kolom id berulang (item_id, dept_id, dst) jauh lebih hemat memori sebagai 'category'
# dibanding string/object biasa. Kolom penjualan harian juga di-downcast ke integer
# sekecil mungkin. Pada dataset sebesar ini perbedaannya bisa mencapai beberapa GB RAM.
for _c in ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']:
    sales[_c] = sales[_c].astype('category')
for _c in half_days:
    sales[_c] = pd.to_numeric(sales[_c], downcast='integer')

print("Perkiraan penggunaan memori 'sales' setelah downcast: "
      f"{sales.memory_usage(deep=True).sum() / 1e6:.1f} MB")


Bentuk data setelah filter & sampling: (15245, 961)
Perkiraan penggunaan memori 'sales' setelah downcast: 20.5 MB


In [5]:
# Ubah ke format long: satu baris = satu item pada satu hari
sales_long = sales.melt(
    id_vars=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name='d',
    value_name='sales'
)

# Gabungkan dengan calendar (untuk minggu / wm_yr_wk) dan sell_prices (untuk harga)
sales_long = sales_long.merge(calendar[['d', 'wm_yr_wk']], on='d', how='left')
sales_long = sales_long.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

# --- Perbaikan anti-crash RAM: downcast hasil merge & buang dataframe perantara ---
sales_long['d'] = sales_long['d'].astype('category')
sales_long['sales'] = pd.to_numeric(sales_long['sales'], downcast='integer')
if 'wm_yr_wk' in sales_long.columns:
    sales_long['wm_yr_wk'] = pd.to_numeric(sales_long['wm_yr_wk'], downcast='integer')
if 'sell_price' in sales_long.columns:
    sales_long['sell_price'] = pd.to_numeric(sales_long['sell_price'], downcast='float')

print("sales_long:", sales_long.shape)
print(f"Perkiraan penggunaan memori 'sales_long': "
      f"{sales_long.memory_usage(deep=True).sum() / 1e6:.1f} MB")
sales_long.head()

# 'sales' (format lebar) sudah tidak dipakai lagi setelah ini -> bebaskan memorinya
del sales
free_memory()


sales_long: (14574220, 9)
Perkiraan penggunaan memori 'sales_long': 1861.9 MB


## 4. Agregasi Mingguan per State x Kategori (`weekly_avg_enriched`)

Data harian per-item diringkas menjadi **total revenue mingguan** untuk tiap kombinasi *state* x *kategori*
(`revenue = sales x sell_price`, dijumlahkan per minggu), lalu diperkaya dengan fitur kalender `wday` dan `month`.
Fitur-fitur ini yang nantinya jadi input model 1D-CNN dan target peramalan ARIMA.


In [6]:
sl = sales_long.copy()
sl['sales'] = pd.to_numeric(sl['sales'], errors='coerce').fillna(0)
sl['sell_price'] = pd.to_numeric(sl['sell_price'], errors='coerce')
sl['revenue_row'] = sl['sales'] * sl['sell_price'].fillna(0)

weekly_avg = (
    sl.groupby(['wm_yr_wk', 'state_id', 'cat_id'], as_index=False)
      .agg(revenue=('revenue_row', 'sum'))
)

calendar_features = calendar[['wm_yr_wk', 'wday', 'month']].drop_duplicates('wm_yr_wk')
weekly_avg_enriched = (
    weekly_avg.merge(calendar_features, on='wm_yr_wk', how='left')
              .sort_values(['state_id', 'cat_id', 'wm_yr_wk'])
              .reset_index(drop=True)
)

print("weekly_avg_enriched:", weekly_avg_enriched.shape)
weekly_avg_enriched.head()


weekly_avg_enriched: (1233, 6)


,wm_yr_wk,state_id,cat_id,revenue,wday,month
0,11101,CA,FOODS,59076.039062,1,1
1,11102,CA,FOODS,68650.531250,1,2
2,11103,CA,FOODS,61515.410156,1,2
3,11104,CA,FOODS,56732.148438,1,2
4,11105,CA,FOODS,60933.789062,1,2


## 5. Konfigurasi Eksperimen

Konfigurasi ini dipakai **sama persis** untuk 1D-CNN, RNN, dan ARIMA supaya perbandingan adil.
`WEIGHT_WEEKS` dipakai khusus untuk menghitung bobot tiap deret pada **WRMSSE** (lihat Bagian 6).

In [7]:
TIME_STEP    = 10   # panjang jendela input (minggu)
TEST_WEEKS   = 30   # panjang periode uji walk-forward (minggu), otomatis menyesuaikan jika data lebih pendek
NUM_FEATURES = 3    # revenue, wday, month
CNN_EPOCHS   = 15   # epoch training CNN di setiap langkah walk-forward (retrain tiap minggu)
CNN_BATCH    = 8
RNN_EPOCHS   = 15   # SAMA dengan CNN_EPOCHS -> agar protokol training CNN vs RNN adil
RNN_BATCH    = 8    # SAMA dengan CNN_BATCH  -> agar protokol training CNN vs RNN adil
LSTM_EPOCHS  = 15   # SAMA dengan CNN/RNN_EPOCHS -> agar protokol training LSTM adil terhadap model lain
LSTM_BATCH   = 8    # SAMA dengan CNN/RNN_BATCH
GRU_EPOCHS   = 15   # SAMA dengan CNN/RNN_EPOCHS -> agar protokol training GRU adil terhadap model lain
GRU_BATCH    = 8    # SAMA dengan CNN/RNN_BATCH
DNN_EPOCHS   = 15   # SAMA dengan CNN/RNN_EPOCHS -> agar protokol training DNN adil terhadap model lain
DNN_BATCH    = 8    # SAMA dengan CNN/RNN_BATCH
WEIGHT_WEEKS = 4     # jumlah minggu terakhir periode latih yang dipakai utk menghitung bobot WRMSSE
                     # (analog dengan bobot resmi M5 yang berbasis ~28 hari/4 minggu terakhir data latih)

print(f"TIME_STEP={TIME_STEP}, TEST_WEEKS={TEST_WEEKS}, NUM_FEATURES={NUM_FEATURES}, "
      f"CNN_EPOCHS={CNN_EPOCHS}, CNN_BATCH={CNN_BATCH}, RNN_EPOCHS={RNN_EPOCHS}, RNN_BATCH={RNN_BATCH}, "
      f"LSTM_EPOCHS={LSTM_EPOCHS}, LSTM_BATCH={LSTM_BATCH}, GRU_EPOCHS={GRU_EPOCHS}, GRU_BATCH={GRU_BATCH}, "
      f"DNN_EPOCHS={DNN_EPOCHS}, DNN_BATCH={DNN_BATCH}, WEIGHT_WEEKS={WEIGHT_WEEKS}")


TIME_STEP=10, TEST_WEEKS=30, NUM_FEATURES=3, CNN_EPOCHS=15, CNN_BATCH=8, RNN_EPOCHS=15, RNN_BATCH=8, LSTM_EPOCHS=15, LSTM_BATCH=8, GRU_EPOCHS=15, GRU_BATCH=8, DNN_EPOCHS=15, DNN_BATCH=8, WEIGHT_WEEKS=4


## 6. Fungsi Bantu

- `create_dataset`: mengubah deret waktu menjadi dataset *supervised* dengan jendela geser (*sliding window*) — dipakai untuk menyiapkan input 1D-CNN & RNN.
- `rmsse_score`: menghitung **RMSSE per deret** sesuai definisi resmi M5 — RMSE forecast dibagi RMSE naive
  one-step-ahead yang dihitung dari **seluruh riwayat data latih** deret tersebut (bukan cuma beberapa minggu
  terakhir), supaya skalanya tidak "mengintip" data uji dan identik untuk semua model.
- `series_weight`: menghitung bobot mentah tiap deret untuk **WRMSSE**, berdasarkan total revenue deret
  tersebut pada `WEIGHT_WEEKS` minggu terakhir periode latih. Bobot ini dinormalisasi (dibagi total semua
  deret) sebelum dipakai merata-ratakan RMSSE secara berbobot di Bagian 11.

In [8]:
def create_dataset(data, time_step=10, num_features=None):
    """Ubah data time series menjadi dataset supervised dengan fitur lag (sliding window).

    Args:
        data: array (n_samples, n_features).
        time_step: panjang jendela input (jumlah langkah waktu sebelumnya).
        num_features: jumlah fitur yang diambil untuk X (default: semua kolom).

    Returns:
        X, y: fitur (n, time_step, num_features) dan target (n,) -> target = kolom ke-0 (revenue).
    """
    if num_features is None:
        num_features = data.shape[1]
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:(i + time_step), :num_features])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


def rmsse_score(y_true, y_pred, train_history):
    """RMSSE (Root Mean Squared Scaled Error) sesuai definisi resmi M5.

    RMSSE = RMSE(forecast) / RMSE(naive one-step-ahead pada SELURUH data latih).

    `train_history` harus berisi seluruh nilai revenue pada periode LATIH saja (sebelum test_start),
    supaya skala pembagi identik dengan definisi resmi M5 dan tidak pernah menyentuh data uji.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    hist = np.asarray(train_history, dtype=float)
    diffs = np.diff(hist)
    if len(diffs) == 0:
        return rmse
    scale = np.sqrt(np.mean(diffs ** 2))
    return rmse if scale == 0 else rmse / scale


def series_weight(train_history, weight_weeks):
    """Bobot mentah sebuah deret untuk WRMSSE = total revenue deret ini pada `weight_weeks` minggu
    terakhir periode latih (analog bobot resmi M5 yang berbasis dollar sales beberapa minggu terakhir
    sebelum periode uji). Dinormalisasi belakangan supaya total bobot semua deret = 1.
    """
    hist = np.asarray(train_history, dtype=float)
    window = hist[-weight_weeks:] if len(hist) >= weight_weeks else hist
    return float(np.sum(window))


## 7. Tuning Hyperparameter — 1D-CNN (Keras Tuner)

Tuning dilakukan **satu kali** pada deret representatif (`HOBBIES`-`CA`) memakai `RandomSearch` dari Keras Tuner,
lalu hyperparameter terbaik dipakai untuk semua deret pada tahap walk-forward (Bagian 10).

In [9]:
%pip install -q keras-tuner   # jalankan sekali jika belum terpasang
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

tf.random.set_seed(SEED)


def build_model(hp):
    inputs = Input(shape=(TIME_STEP, NUM_FEATURES))
    x = inputs
    num_cnn_layers = hp.Int('num_cnn_layers', min_value=1, max_value=3, step=1)
    for i in range(num_cnn_layers):
        num_filters = hp.Int(f'filters_{i}', min_value=32, max_value=128, step=32)
        kernel_size = hp.Choice(f'kernel_size_{i}', values=[2, 3, 5])
        x = Conv1D(filters=num_filters, kernel_size=kernel_size,
                   activation='relu', padding='causal')(x)
    x = Flatten()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model


# --- data tuning: deret representatif HOBBIES-CA ---
tuning_series = weekly_avg_enriched[
    (weekly_avg_enriched['cat_id'] == 'HOBBIES') &
    (weekly_avg_enriched['state_id'] == 'CA')
].sort_values('wm_yr_wk')

feat_tune = tuning_series[['revenue', 'wday', 'month']].values
scaler_tune = MinMaxScaler()
feat_tune_scaled = scaler_tune.fit_transform(feat_tune)
X_tune, y_tune = create_dataset(feat_tune_scaled, TIME_STEP, NUM_FEATURES)
print("Bentuk data tuning:", X_tune.shape)

tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=15,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='cnn_walkforward_tuning',
    overwrite=True,
)

print("Menjalankan hyperparameter search untuk 1D-CNN...")
tuner.search(X_tune, y_tune, epochs=15, batch_size=16, validation_split=0.2, verbose=0)
print("Selesai.")

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_hp_values = dict(best_hps.values)
print("\n=== Hyperparameter 1D-CNN terbaik ===")
for k, v in best_hp_values.items():
    print(f"{k}: {v}")

# --- catat SEMUA trial (untuk dilampirkan di laporan) ---
trial_records = []
for trial_id, trial in tuner.oracle.trials.items():
    rec = dict(trial.hyperparameters.values)
    rec['trial_id'] = trial_id
    rec['val_loss'] = trial.score
    trial_records.append(rec)

cnn_tuning_results = pd.DataFrame(trial_records).sort_values('val_loss').reset_index(drop=True)
cnn_tuning_results.insert(0, 'rank', range(1, len(cnn_tuning_results) + 1))

print("\n=== Log Lengkap Hyperparameter Tuning 1D-CNN (semua trial) ===")
display(cnn_tuning_results)

cnn_tuning_results.to_csv('cnn_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: cnn_hyperparameter_tuning_log.csv")

# --- Perbaikan anti-crash RAM: hyperparameter tuning membangun & melatih banyak model
# kecil (max_trials x executions_per_trial). Bebaskan memori sebelum lanjut ke sel berikutnya.
free_memory()
print_ram(f'setelah tuning selesai (sel 16)')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.2 MB/s eta 0:00:00
Bentuk data tuning: (127, 10, 3)
Menjalankan hyperparameter search untuk 1D-CNN...
Selesai.

=== Hyperparameter 1D-CNN terbaik ===
num_cnn_layers: 2
filters_0: 96
kernel_size_0: 2
learning_rate: 0.002254325243540417
filters_1: 128
kernel_size_1: 2
filters_2: 96
kernel_size_2: 2

=== Log Lengkap Hyperparameter Tuning 1D-CNN (semua trial) ===


,rank,num_cnn_layers,filters_0,kernel_size_0,learning_rate,filters_1,kernel_size_1,trial_id,val_loss,filters_2,kernel_size_2
0,1,2,96,2,0.002254,128,2,14,0.032317,96.0,2.0
1,2,1,128,2,0.004947,96,3,06,0.033078,64.0,3.0
2,3,3,64,5,0.001579,96,2,04,0.034752,96.0,3.0
3,4,3,128,2,0.000641,64,2,10,0.035289,32.0,3.0
4,5,3,128,5,0.005224,128,5,09,0.035521,128.0,5.0
5,6,2,96,3,0.001723,128,3,02,0.035634,NaN,NaN
6,7,2,32,5,0.001223,64,3,12,0.035916,64.0,3.0
7,8,3,96,2,0.001078,96,2,05,0.035988,96.0,5.0
8,9,3,96,5,0.004975,128,5,08,0.037268,64.0,5.0
9,10,3,64,3,0.001848,32,2,07,0.037814,64.0,5.0


Disimpan ke: cnn_hyperparameter_tuning_log.csv
[RAM] setelah tuning selesai (sel 16) peak RSS ~ 3755.1 MB


## 8. Tuning Hyperparameter — RNN (Keras Tuner)

Agar perbandingan dengan 1D-CNN **adil**, RNN di-tuning dengan protokol yang **identik**:
deret representatif yang sama (`HOBBIES`-`CA`), jumlah trial sama (15), epoch & `validation_split` sama,
dan ruang pencarian hyperparameter yang sepadan (jumlah layer, jumlah unit, learning rate).
Arsitektur RNN memakai layer `SimpleRNN` (RNN klasik) yang ditumpuk (*stacked*), diakhiri layer `Dense(1)`.

In [10]:
from tensorflow.keras.layers import SimpleRNN


def build_rnn_model(hp):
    inputs = Input(shape=(TIME_STEP, NUM_FEATURES))
    x = inputs
    num_rnn_layers = hp.Int('num_rnn_layers', min_value=1, max_value=3, step=1)
    for i in range(num_rnn_layers):
        units = hp.Int(f'units_{i}', min_value=32, max_value=128, step=32)
        return_seq = i < num_rnn_layers - 1  # hanya layer terakhir yang tidak return_sequences
        x = SimpleRNN(units, activation='tanh', return_sequences=return_seq)(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model


# --- data tuning: SAMA PERSIS dengan CNN (X_tune, y_tune dari deret HOBBIES-CA) ---
rnn_tuner = kt.RandomSearch(
    build_rnn_model,
    objective='val_loss',
    max_trials=15,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='rnn_walkforward_tuning',
    overwrite=True,
)

print("Menjalankan hyperparameter search untuk RNN...")
rnn_tuner.search(X_tune, y_tune, epochs=15, batch_size=16, validation_split=0.2, verbose=0)
print("Selesai.")

best_rnn_hps = rnn_tuner.get_best_hyperparameters(num_trials=1)[0]
best_rnn_hp_values = dict(best_rnn_hps.values)
print("\n=== Hyperparameter RNN terbaik ===")
for k, v in best_rnn_hp_values.items():
    print(f"{k}: {v}")

# --- catat SEMUA trial (untuk dilampirkan di laporan) ---
rnn_trial_records = []
for trial_id, trial in rnn_tuner.oracle.trials.items():
    rec = dict(trial.hyperparameters.values)
    rec['trial_id'] = trial_id
    rec['val_loss'] = trial.score
    rnn_trial_records.append(rec)

rnn_tuning_results = pd.DataFrame(rnn_trial_records).sort_values('val_loss').reset_index(drop=True)
rnn_tuning_results.insert(0, 'rank', range(1, len(rnn_tuning_results) + 1))

print("\n=== Log Lengkap Hyperparameter Tuning RNN (semua trial) ===")
display(rnn_tuning_results)

rnn_tuning_results.to_csv('rnn_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: rnn_hyperparameter_tuning_log.csv")

# --- Perbaikan anti-crash RAM: hyperparameter tuning membangun & melatih banyak model
# kecil (max_trials x executions_per_trial). Bebaskan memori sebelum lanjut ke sel berikutnya.
free_memory()
print_ram(f'setelah tuning selesai (sel 18)')


Menjalankan hyperparameter search untuk RNN...
Selesai.

=== Hyperparameter RNN terbaik ===
num_rnn_layers: 1
units_0: 96
learning_rate: 0.008053623177641262
units_1: 128

=== Log Lengkap Hyperparameter Tuning RNN (semua trial) ===


,rank,num_rnn_layers,units_0,learning_rate,units_1,trial_id,val_loss,units_2
0,1,1,96,0.008054,128,02,0.027916,NaN
1,2,2,96,0.003926,64,01,0.029071,NaN
2,3,3,32,0.008961,32,03,0.030037,32.0
3,4,3,128,0.000679,32,12,0.030912,32.0
4,5,3,128,0.000836,32,14,0.032226,64.0
5,6,3,32,0.000113,32,10,0.037230,64.0
6,7,1,32,0.001154,32,09,0.037674,64.0
7,8,1,96,0.000542,64,07,0.041986,128.0
8,9,2,64,0.006846,32,00,0.045057,NaN
9,10,3,64,0.000770,32,13,0.048717,128.0


Disimpan ke: rnn_hyperparameter_tuning_log.csv
[RAM] setelah tuning selesai (sel 18) peak RSS ~ 3755.1 MB


## 9. Tuning Hyperparameter — LSTM (Keras Tuner)

Sama seperti RNN, LSTM di-tuning dengan protokol yang **identik**: deret representatif yang sama
(`HOBBIES`-`CA`), data tuning yang sama (`X_tune`, `y_tune`), jumlah trial sama (15), epoch &
`validation_split` sama, dan ruang pencarian hyperparameter yang sepadan (jumlah layer, jumlah unit,
learning rate). Arsitektur memakai layer `LSTM` yang ditumpuk (*stacked*), diakhiri layer `Dense(1)`.

In [ ]:
from tensorflow.keras.layers import LSTM


def build_lstm_model(hp):
    inputs = Input(shape=(TIME_STEP, NUM_FEATURES))
    x = inputs
    num_lstm_layers = hp.Int('num_lstm_layers', min_value=1, max_value=3, step=1)
    for i in range(num_lstm_layers):
        units = hp.Int(f'units_{i}', min_value=32, max_value=128, step=32)
        return_seq = i < num_lstm_layers - 1  # hanya layer terakhir yang tidak return_sequences
        x = LSTM(units, activation='tanh', return_sequences=return_seq)(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model


# --- data tuning: SAMA PERSIS dengan CNN/RNN (X_tune, y_tune dari deret HOBBIES-CA) ---
lstm_tuner = kt.RandomSearch(
    build_lstm_model,
    objective='val_loss',
    max_trials=15,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='lstm_walkforward_tuning',
    overwrite=True,
)

print("Menjalankan hyperparameter search untuk LSTM...")
lstm_tuner.search(X_tune, y_tune, epochs=15, batch_size=16, validation_split=0.2, verbose=0)
print("Selesai.")

best_lstm_hps = lstm_tuner.get_best_hyperparameters(num_trials=1)[0]
best_lstm_hp_values = dict(best_lstm_hps.values)
print("\n=== Hyperparameter LSTM terbaik ===")
for k, v in best_lstm_hp_values.items():
    print(f"{k}: {v}")

# --- catat SEMUA trial (untuk dilampirkan di laporan) ---
lstm_trial_records = []
for trial_id, trial in lstm_tuner.oracle.trials.items():
    rec = dict(trial.hyperparameters.values)
    rec['trial_id'] = trial_id
    rec['val_loss'] = trial.score
    lstm_trial_records.append(rec)

lstm_tuning_results = pd.DataFrame(lstm_trial_records).sort_values('val_loss').reset_index(drop=True)
lstm_tuning_results.insert(0, 'rank', range(1, len(lstm_tuning_results) + 1))

print("\n=== Log Lengkap Hyperparameter Tuning LSTM (semua trial) ===")
display(lstm_tuning_results)

lstm_tuning_results.to_csv('lstm_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: lstm_hyperparameter_tuning_log.csv")

# --- Perbaikan anti-crash RAM: hyperparameter tuning membangun & melatih banyak model
# kecil (max_trials x executions_per_trial). Bebaskan memori sebelum lanjut ke sel berikutnya.
free_memory()
print_ram(f'setelah tuning selesai (sel 20)')


Menjalankan hyperparameter search untuk LSTM...


## 10. Tuning Hyperparameter — GRU (Keras Tuner)

Protokol tuning GRU juga **identik** dengan CNN/RNN/LSTM di atas: deret representatif yang sama,
data tuning yang sama, jumlah trial, epoch, dan `validation_split` yang sama. Arsitektur memakai
layer `GRU` yang ditumpuk (*stacked*), diakhiri layer `Dense(1)`.

In [ ]:
from tensorflow.keras.layers import GRU


def build_gru_model(hp):
    inputs = Input(shape=(TIME_STEP, NUM_FEATURES))
    x = inputs
    num_gru_layers = hp.Int('num_gru_layers', min_value=1, max_value=3, step=1)
    for i in range(num_gru_layers):
        units = hp.Int(f'units_{i}', min_value=32, max_value=128, step=32)
        return_seq = i < num_gru_layers - 1  # hanya layer terakhir yang tidak return_sequences
        x = GRU(units, activation='tanh', return_sequences=return_seq)(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model


# --- data tuning: SAMA PERSIS dengan CNN/RNN/LSTM (X_tune, y_tune dari deret HOBBIES-CA) ---
gru_tuner = kt.RandomSearch(
    build_gru_model,
    objective='val_loss',
    max_trials=15,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='gru_walkforward_tuning',
    overwrite=True,
)

print("Menjalankan hyperparameter search untuk GRU...")
gru_tuner.search(X_tune, y_tune, epochs=15, batch_size=16, validation_split=0.2, verbose=0)
print("Selesai.")

best_gru_hps = gru_tuner.get_best_hyperparameters(num_trials=1)[0]
best_gru_hp_values = dict(best_gru_hps.values)
print("\n=== Hyperparameter GRU terbaik ===")
for k, v in best_gru_hp_values.items():
    print(f"{k}: {v}")

# --- catat SEMUA trial (untuk dilampirkan di laporan) ---
gru_trial_records = []
for trial_id, trial in gru_tuner.oracle.trials.items():
    rec = dict(trial.hyperparameters.values)
    rec['trial_id'] = trial_id
    rec['val_loss'] = trial.score
    gru_trial_records.append(rec)

gru_tuning_results = pd.DataFrame(gru_trial_records).sort_values('val_loss').reset_index(drop=True)
gru_tuning_results.insert(0, 'rank', range(1, len(gru_tuning_results) + 1))

print("\n=== Log Lengkap Hyperparameter Tuning GRU (semua trial) ===")
display(gru_tuning_results)

gru_tuning_results.to_csv('gru_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: gru_hyperparameter_tuning_log.csv")

# --- Perbaikan anti-crash RAM: hyperparameter tuning membangun & melatih banyak model
# kecil (max_trials x executions_per_trial). Bebaskan memori sebelum lanjut ke sel berikutnya.
free_memory()
print_ram(f'setelah tuning selesai (sel 22)')


## 11. Tuning Hyperparameter — DNN (Keras Tuner)

DNN dipakai sebagai *baseline* non-sequential: jendela input (`TIME_STEP` x `NUM_FEATURES`) di-*flatten*
menjadi satu vektor lalu diproses dengan tumpukan layer `Dense`. Protokol tuning tetap **identik** dengan
model lain: deret representatif yang sama, data tuning yang sama, jumlah trial, epoch, dan
`validation_split` yang sama.

In [ ]:
def build_dnn_model(hp):
    inputs = Input(shape=(TIME_STEP, NUM_FEATURES))
    x = Flatten()(inputs)
    num_dnn_layers = hp.Int('num_dnn_layers', min_value=1, max_value=3, step=1)
    for i in range(num_dnn_layers):
        units = hp.Int(f'dense_units_{i}', min_value=32, max_value=128, step=32)
        x = Dense(units, activation='relu')(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model


# --- data tuning: SAMA PERSIS dengan model lain (X_tune, y_tune dari deret HOBBIES-CA) ---
dnn_tuner = kt.RandomSearch(
    build_dnn_model,
    objective='val_loss',
    max_trials=15,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='dnn_walkforward_tuning',
    overwrite=True,
)

print("Menjalankan hyperparameter search untuk DNN...")
dnn_tuner.search(X_tune, y_tune, epochs=15, batch_size=16, validation_split=0.2, verbose=0)
print("Selesai.")

best_dnn_hps = dnn_tuner.get_best_hyperparameters(num_trials=1)[0]
best_dnn_hp_values = dict(best_dnn_hps.values)
print("\n=== Hyperparameter DNN terbaik ===")
for k, v in best_dnn_hp_values.items():
    print(f"{k}: {v}")

# --- catat SEMUA trial (untuk dilampirkan di laporan) ---
dnn_trial_records = []
for trial_id, trial in dnn_tuner.oracle.trials.items():
    rec = dict(trial.hyperparameters.values)
    rec['trial_id'] = trial_id
    rec['val_loss'] = trial.score
    dnn_trial_records.append(rec)

dnn_tuning_results = pd.DataFrame(dnn_trial_records).sort_values('val_loss').reset_index(drop=True)
dnn_tuning_results.insert(0, 'rank', range(1, len(dnn_tuning_results) + 1))

print("\n=== Log Lengkap Hyperparameter Tuning DNN (semua trial) ===")
display(dnn_tuning_results)

dnn_tuning_results.to_csv('dnn_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: dnn_hyperparameter_tuning_log.csv")

# --- Perbaikan anti-crash RAM: hyperparameter tuning membangun & melatih banyak model
# kecil (max_trials x executions_per_trial). Bebaskan memori sebelum lanjut ke sel berikutnya.
free_memory()
print_ram(f'setelah tuning selesai (sel 24)')


## 12. Tuning Hyperparameter — ARIMA (Grid Search Orde p, d, q)

Untuk ARIMA, "hyperparameter" yang di-tuning adalah orde `(p, d, q)` dengan batas `p <= 2, d <= 1, q <= 2`.
Grid search berbasis **AIC**, dilakukan per deret (state x kategori) pada data latih awal saja (sebelum
periode uji walk-forward) — konsisten dengan 1D-CNN/RNN/LSTM/GRU/DNN yang juga hanya di-tuning memakai data sebelum
periode uji.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA


def arima_order_search(train_series, p_max=2, d_max=1, q_max=2):
    """Grid search orde ARIMA berbasis AIC pada data latih awal (sebelum periode uji walk-forward)."""
    best_aic = np.inf
    best_order = (0, 1, 0)
    records = []
    for p, d, q in itertools.product(range(p_max + 1), range(d_max + 1), range(q_max + 1)):
        try:
            fit = ARIMA(train_series, order=(p, d, q)).fit()
            records.append({'p': p, 'd': d, 'q': q, 'AIC': fit.aic})
            if fit.aic < best_aic:
                best_aic = fit.aic
                best_order = (p, d, q)
        except Exception:
            continue
    return best_order, best_aic, pd.DataFrame(records)


arima_best_orders = []
arima_all_trials = []

for cat in CATEGORIES:
    for state in STATES:
        series_df = weekly_avg_enriched[
            (weekly_avg_enriched['cat_id'] == cat) &
            (weekly_avg_enriched['state_id'] == state)
        ].sort_values('wm_yr_wk')

        if len(series_df) < TIME_STEP + 20:
            print(f"Lewati {cat}-{state}: data terlalu pendek ({len(series_df)} minggu)")
            continue

        n_test = min(TEST_WEEKS, len(series_df) - TIME_STEP - 10)
        test_start = len(series_df) - n_test
        initial_train_rev = series_df['revenue'].values[:test_start]

        order, aic, trials_df = arima_order_search(initial_train_rev)
        trials_df['cat_id'] = cat
        trials_df['state_id'] = state
        arima_all_trials.append(trials_df)

        arima_best_orders.append({
            'cat_id': cat, 'state_id': state,
            'best_p': order[0], 'best_d': order[1], 'best_q': order[2],
            'AIC': aic, 'n_test_weeks': n_test
        })
        print(f"{cat:10s} {state} -> orde terbaik ARIMA(p,d,q)={order}, AIC={aic:.2f}")

arima_best_orders = pd.DataFrame(arima_best_orders)
arima_tuning_results_all = pd.concat(arima_all_trials, ignore_index=True)

print("\n=== Orde ARIMA Terbaik per Deret ===")
display(arima_best_orders)

print("\n=== Log Lengkap Grid Search ARIMA (semua kombinasi p,d,q per deret) ===")
display(arima_tuning_results_all.sort_values(['cat_id', 'state_id', 'AIC']))

arima_best_orders.to_csv('arima_best_orders.csv', index=False)
arima_tuning_results_all.to_csv('arima_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: arima_best_orders.csv dan arima_hyperparameter_tuning_log.csv")


## 13. Evaluasi Walk-Forward yang Adil (1D-CNN vs RNN vs LSTM vs GRU vs DNN vs ARIMA)

**Protokol (identik untuk keenam model):**
1. Setiap deret mingguan dibagi: `TEST_WEEKS` minggu terakhir jadi periode uji *walk-forward*; sisanya periode latih.
2. Pada setiap langkah walk-forward, keenam model **di-retrain ulang** memakai *expanding window* (semua data
   sampai sebelum minggu yang diramal), lalu meramalkan **1 minggu ke depan**.
3. 1D-CNN, RNN, LSTM, GRU, dan DNN memakai fitur input yang **sama persis** (`revenue`, `wday`, `month`),
   jendela waktu yang sama (`TIME_STEP`), dan `scaler` yang sama (di-fit hanya pada data latih).
4. ARIMA memakai deret `revenue` mentah (tanpa scaling) sesuai kelaziman model statistik klasik.

In [ ]:
def build_cnn_from_hp(hp_values, time_step, num_features):
    """Bangun model CNN dari hyperparameter tetap hasil tuning."""
    inputs = Input(shape=(time_step, num_features))
    x = inputs
    for i in range(hp_values['num_cnn_layers']):
        x = Conv1D(
            filters=hp_values[f'filters_{i}'],
            kernel_size=hp_values[f'kernel_size_{i}'],
            activation='relu',
            padding='causal',
        )(x)
    x = Flatten()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(hp_values['learning_rate']))
    return model


def build_rnn_from_hp(hp_values, time_step, num_features):
    """Bangun model RNN dari hyperparameter tetap hasil tuning (analog build_cnn_from_hp)."""
    inputs = Input(shape=(time_step, num_features))
    x = inputs
    num_rnn_layers = hp_values['num_rnn_layers']
    for i in range(num_rnn_layers):
        return_seq = i < num_rnn_layers - 1
        x = SimpleRNN(hp_values[f'units_{i}'], activation='tanh', return_sequences=return_seq)(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(hp_values['learning_rate']))
    return model


def build_lstm_from_hp(hp_values, time_step, num_features):
    """Bangun model LSTM dari hyperparameter tetap hasil tuning (analog build_rnn_from_hp)."""
    inputs = Input(shape=(time_step, num_features))
    x = inputs
    num_lstm_layers = hp_values['num_lstm_layers']
    for i in range(num_lstm_layers):
        return_seq = i < num_lstm_layers - 1
        x = LSTM(hp_values[f'units_{i}'], activation='tanh', return_sequences=return_seq)(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(hp_values['learning_rate']))
    return model


def build_gru_from_hp(hp_values, time_step, num_features):
    """Bangun model GRU dari hyperparameter tetap hasil tuning (analog build_rnn_from_hp)."""
    inputs = Input(shape=(time_step, num_features))
    x = inputs
    num_gru_layers = hp_values['num_gru_layers']
    for i in range(num_gru_layers):
        return_seq = i < num_gru_layers - 1
        x = GRU(hp_values[f'units_{i}'], activation='tanh', return_sequences=return_seq)(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(hp_values['learning_rate']))
    return model


def build_dnn_from_hp(hp_values, time_step, num_features):
    """Bangun model DNN (dense murni pada jendela yang di-flatten) dari hyperparameter tetap hasil tuning."""
    inputs = Input(shape=(time_step, num_features))
    x = Flatten()(inputs)
    num_dnn_layers = hp_values['num_dnn_layers']
    for i in range(num_dnn_layers):
        x = Dense(hp_values[f'dense_units_{i}'], activation='relu')(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(hp_values['learning_rate']))
    return model


def walk_forward_cnn(feature_matrix, time_step, n_test, hp_values, epochs=15, batch_size=8):
    """CNN di-retrain ulang (expanding window) di setiap langkah walk-forward."""
    n = len(feature_matrix)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        train_data = feature_matrix[:i]
        X_train, y_train = create_dataset(train_data, time_step, feature_matrix.shape[1])
        if len(X_train) < 5:
            preds.append(train_data[-1, 0])
            continue
        model = build_cnn_from_hp(hp_values, time_step, feature_matrix.shape[1])
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        X_input = feature_matrix[i - time_step:i].reshape(1, time_step, feature_matrix.shape[1])
        pred = model(X_input, training=False).numpy()[0, 0]
        preds.append(pred)
        free_memory(model)
    return np.array(preds)


def walk_forward_rnn(feature_matrix, time_step, n_test, hp_values, epochs=15, batch_size=8):
    """RNN di-retrain ulang (expanding window) di setiap langkah walk-forward -- protokol IDENTIK
    dengan walk_forward_cnn di atas (perbedaan hanya pada arsitektur model)."""
    n = len(feature_matrix)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        train_data = feature_matrix[:i]
        X_train, y_train = create_dataset(train_data, time_step, feature_matrix.shape[1])
        if len(X_train) < 5:
            preds.append(train_data[-1, 0])
            continue
        model = build_rnn_from_hp(hp_values, time_step, feature_matrix.shape[1])
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        X_input = feature_matrix[i - time_step:i].reshape(1, time_step, feature_matrix.shape[1])
        pred = model(X_input, training=False).numpy()[0, 0]
        preds.append(pred)
        free_memory(model)
    return np.array(preds)


def walk_forward_lstm(feature_matrix, time_step, n_test, hp_values, epochs=15, batch_size=8):
    """LSTM di-retrain ulang (expanding window) di setiap langkah walk-forward -- protokol IDENTIK
    dengan walk_forward_cnn/rnn di atas (perbedaan hanya pada arsitektur model)."""
    n = len(feature_matrix)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        train_data = feature_matrix[:i]
        X_train, y_train = create_dataset(train_data, time_step, feature_matrix.shape[1])
        if len(X_train) < 5:
            preds.append(train_data[-1, 0])
            continue
        model = build_lstm_from_hp(hp_values, time_step, feature_matrix.shape[1])
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        X_input = feature_matrix[i - time_step:i].reshape(1, time_step, feature_matrix.shape[1])
        pred = model(X_input, training=False).numpy()[0, 0]
        preds.append(pred)
        free_memory(model)
    return np.array(preds)


def walk_forward_gru(feature_matrix, time_step, n_test, hp_values, epochs=15, batch_size=8):
    """GRU di-retrain ulang (expanding window) di setiap langkah walk-forward -- protokol IDENTIK
    dengan walk_forward_cnn/rnn/lstm di atas (perbedaan hanya pada arsitektur model)."""
    n = len(feature_matrix)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        train_data = feature_matrix[:i]
        X_train, y_train = create_dataset(train_data, time_step, feature_matrix.shape[1])
        if len(X_train) < 5:
            preds.append(train_data[-1, 0])
            continue
        model = build_gru_from_hp(hp_values, time_step, feature_matrix.shape[1])
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        X_input = feature_matrix[i - time_step:i].reshape(1, time_step, feature_matrix.shape[1])
        pred = model(X_input, training=False).numpy()[0, 0]
        preds.append(pred)
        free_memory(model)
    return np.array(preds)


def walk_forward_dnn(feature_matrix, time_step, n_test, hp_values, epochs=15, batch_size=8):
    """DNN di-retrain ulang (expanding window) di setiap langkah walk-forward -- protokol IDENTIK
    dengan model lain di atas (perbedaan hanya pada arsitektur model)."""
    n = len(feature_matrix)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        train_data = feature_matrix[:i]
        X_train, y_train = create_dataset(train_data, time_step, feature_matrix.shape[1])
        if len(X_train) < 5:
            preds.append(train_data[-1, 0])
            continue
        model = build_dnn_from_hp(hp_values, time_step, feature_matrix.shape[1])
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        X_input = feature_matrix[i - time_step:i].reshape(1, time_step, feature_matrix.shape[1])
        pred = model(X_input, training=False).numpy()[0, 0]
        preds.append(pred)
        free_memory(model)
    return np.array(preds)


def walk_forward_arima(series_1d, order, n_test):
    """ARIMA di-fit ulang (expanding window) di setiap langkah walk-forward, orde tetap."""
    n = len(series_1d)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        hist = series_1d[:i]
        try:
            fc = ARIMA(hist, order=order).fit().forecast(steps=1).iloc[0]
        except Exception:
            fc = hist[-1]
        preds.append(fc)
    return np.array(preds)


print("Fungsi walk-forward CNN, RNN, LSTM, GRU, DNN & ARIMA siap dipakai.")


In [ ]:
# --- Perbaikan anti-crash RAM: RESUME dari checkpoint ---
# Jika sesi sebelumnya crash, file partial di bawah ini akan berisi hasil kombinasi
# kategori x state yang SUDAH selesai. Kita muat ulang supaya tidak perlu mengulang dari nol.
_partial_path = 'walkforward_cnn_rnn_lstm_gru_dnn_arima_results_partial.csv'
_weights_partial_path = 'series_weights_wrmsse_partial.csv'
_expected_models = {'1D-CNN', 'RNN', 'LSTM', 'GRU', 'DNN', 'ARIMA'}

if os.path.exists(_partial_path):
    _partial_df = pd.read_csv(_partial_path)
    walkforward_results = _partial_df.to_dict('records')
    print(f"[resume] Memuat {len(walkforward_results)} baris hasil dari checkpoint sebelumnya.")
else:
    walkforward_results = []

if os.path.exists(_weights_partial_path):
    series_weights = pd.read_csv(_weights_partial_path).to_dict('records')
    print(f"[resume] Memuat {len(series_weights)} bobot deret dari checkpoint sebelumnya.")
else:
    series_weights = []

# Kombinasi (Category, State) yang SUDAH lengkap (keenam model sudah punya hasil)
_completed_combos = set()
if walkforward_results:
    _tmp = pd.DataFrame(walkforward_results)
    for (c, s), grp in _tmp.groupby(['Category', 'State']):
        if _expected_models.issubset(set(grp['Model'])):
            _completed_combos.add((c, s))
    if _completed_combos:
        print(f"[resume] Kombinasi yang akan DILEWATI karena sudah selesai: {sorted(_completed_combos)}")

walkforward_predictions = {}  # simpan untuk plotting ulang jika perlu (tidak ikut di-resume, hanya untuk sesi berjalan)

for cat in CATEGORIES:
    for state in STATES:
        if (cat, state) in _completed_combos:
            print(f"Lewati {cat}-{state}: sudah selesai di checkpoint sebelumnya (resume)")
            continue

        series_df = weekly_avg_enriched[
            (weekly_avg_enriched['cat_id'] == cat) &
            (weekly_avg_enriched['state_id'] == state)
        ].sort_values('wm_yr_wk').reset_index(drop=True)

        if len(series_df) < TIME_STEP + 20:
            print(f"Lewati {cat}-{state}: data terlalu pendek")
            continue

        n_test = min(TEST_WEEKS, len(series_df) - TIME_STEP - 10)
        test_start = len(series_df) - n_test

        order_row = arima_best_orders[
            (arima_best_orders['cat_id'] == cat) & (arima_best_orders['state_id'] == state)
        ]
        if order_row.empty:
            print(f"Lewati {cat}-{state}: orde ARIMA belum di-tuning")
            continue
        order = (int(order_row['best_p'].iloc[0]), int(order_row['best_d'].iloc[0]), int(order_row['best_q'].iloc[0]))

        # --- fitur untuk CNN/RNN/LSTM/GRU/DNN (revenue, wday, month) ---
        # PENTING (perbaikan fairness): scaler HANYA di-fit pada periode LATIH (sebelum test_start),
        # baru dipakai men-transform seluruh deret (termasuk periode uji). Sebelumnya scaler di-fit
        # pada seluruh deret (termasuk data uji) -> data leakage yang menguntungkan model neural
        # network secara tidak adil dibanding ARIMA (yang sama sekali tidak pernah "melihat" data uji).
        feat = series_df[['revenue', 'wday', 'month']].values
        scaler = MinMaxScaler()
        scaler.fit(feat[:test_start])
        feat_scaled = scaler.transform(feat)

        train_revenue_hist = series_df['revenue'].values[:test_start]

        print(f"\n>>> {cat} - {state} | n_test={n_test} minggu | orde ARIMA={order}")

        # --- 1D-CNN ---
        cnn_pred_scaled = walk_forward_cnn(
            feat_scaled, TIME_STEP, n_test, best_hp_values,
            epochs=CNN_EPOCHS, batch_size=CNN_BATCH
        )
        dummy = np.zeros((len(cnn_pred_scaled), NUM_FEATURES))
        dummy[:, 0] = cnn_pred_scaled
        cnn_pred = scaler.inverse_transform(dummy)[:, 0]

        # --- RNN (protokol IDENTIK dengan CNN: fitur sama, scaler sama, epoch/batch sama) ---
        rnn_pred_scaled = walk_forward_rnn(
            feat_scaled, TIME_STEP, n_test, best_rnn_hp_values,
            epochs=RNN_EPOCHS, batch_size=RNN_BATCH
        )
        dummy_rnn = np.zeros((len(rnn_pred_scaled), NUM_FEATURES))
        dummy_rnn[:, 0] = rnn_pred_scaled
        rnn_pred = scaler.inverse_transform(dummy_rnn)[:, 0]

        # --- LSTM (protokol IDENTIK dengan CNN/RNN) ---
        lstm_pred_scaled = walk_forward_lstm(
            feat_scaled, TIME_STEP, n_test, best_lstm_hp_values,
            epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH
        )
        dummy_lstm = np.zeros((len(lstm_pred_scaled), NUM_FEATURES))
        dummy_lstm[:, 0] = lstm_pred_scaled
        lstm_pred = scaler.inverse_transform(dummy_lstm)[:, 0]

        # --- GRU (protokol IDENTIK dengan CNN/RNN/LSTM) ---
        gru_pred_scaled = walk_forward_gru(
            feat_scaled, TIME_STEP, n_test, best_gru_hp_values,
            epochs=GRU_EPOCHS, batch_size=GRU_BATCH
        )
        dummy_gru = np.zeros((len(gru_pred_scaled), NUM_FEATURES))
        dummy_gru[:, 0] = gru_pred_scaled
        gru_pred = scaler.inverse_transform(dummy_gru)[:, 0]

        # --- DNN (protokol IDENTIK dengan model lain) ---
        dnn_pred_scaled = walk_forward_dnn(
            feat_scaled, TIME_STEP, n_test, best_dnn_hp_values,
            epochs=DNN_EPOCHS, batch_size=DNN_BATCH
        )
        dummy_dnn = np.zeros((len(dnn_pred_scaled), NUM_FEATURES))
        dummy_dnn[:, 0] = dnn_pred_scaled
        dnn_pred = scaler.inverse_transform(dummy_dnn)[:, 0]

        # --- ARIMA ---
        arima_pred = walk_forward_arima(series_df['revenue'].values, order, n_test)

        actual = series_df['revenue'].values[test_start:]

        weight = series_weight(train_revenue_hist, WEIGHT_WEEKS)
        series_weights.append({'Category': cat, 'State': state, 'Weight_raw': weight})

        walkforward_predictions[(cat, state)] = {
            'actual': actual, 'cnn': cnn_pred, 'rnn': rnn_pred,
            'lstm': lstm_pred, 'gru': gru_pred, 'dnn': dnn_pred, 'arima': arima_pred
        }

        for model_name, pred in [
            ('1D-CNN', cnn_pred), ('RNN', rnn_pred), ('LSTM', lstm_pred),
            ('GRU', gru_pred), ('DNN', dnn_pred), ('ARIMA', arima_pred),
        ]:
            mae = mean_absolute_error(actual, pred)
            rmse = np.sqrt(mean_squared_error(actual, pred))
            mape = np.mean(np.abs((actual - pred) / (actual + 1e-8))) * 100
            rmsse = rmsse_score(actual, pred, train_revenue_hist)

            walkforward_results.append({
                'Category': cat, 'State': state, 'Model': model_name,
                'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'RMSSE': rmsse
            })
            print(f"  {model_name:7s} -> MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%  RMSSE={rmsse:.4f}")

        # Bersihkan sisa memori setelah 1 kombinasi kategori x state selesai
        free_memory()
        print_ram(f'setelah kombinasi {cat}-{state}')

        # --- plot actual vs CNN vs RNN vs LSTM vs GRU vs DNN vs ARIMA ---
        plt.figure(figsize=(10, 4))
        plt.plot(actual, label='Actual', linewidth=2)
        plt.plot(cnn_pred, '--', label='1D-CNN (walk-forward)')
        plt.plot(rnn_pred, '-.', label='RNN (walk-forward)')
        plt.plot(lstm_pred, '--', label='LSTM (walk-forward)')
        plt.plot(gru_pred, '-.', label='GRU (walk-forward)')
        plt.plot(dnn_pred, ':', label='DNN (walk-forward)')
        plt.plot(arima_pred, ':', label='ARIMA (walk-forward)')
        plt.title(f"Walk-Forward Forecast: {cat} - {state}")
        plt.xlabel("Test Week")
        plt.ylabel("Revenue")
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.close('all')  # cegah penumpukan figure di memori selama loop 9 kombinasi

        # --- plot per-model: masing-masing model dibandingkan Actual secara terpisah ---
        model_preds_for_plot = [
            ('1D-CNN', cnn_pred, 'tab:blue'),
            ('RNN',    rnn_pred, 'tab:orange'),
            ('LSTM',   lstm_pred, 'tab:green'),
            ('GRU',    gru_pred, 'tab:red'),
            ('DNN',    dnn_pred, 'tab:purple'),
            ('ARIMA',  arima_pred, 'tab:brown'),
        ]
        fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)
        for ax, (model_name, pred, color) in zip(axes.flatten(), model_preds_for_plot):
            model_rmsse = rmsse_score(actual, pred, train_revenue_hist)
            ax.plot(actual, label='Actual', linewidth=2, color='black')
            ax.plot(pred, '--', label=model_name, color=color)
            ax.set_title(f"{model_name} (RMSSE={model_rmsse:.3f})")
            ax.set_xlabel("Test Week")
            ax.set_ylabel("Revenue")
            ax.legend(loc='best', fontsize=8)
        fig.suptitle(f"Walk-Forward per Model: {cat} - {state}", fontsize=13)
        plt.tight_layout()
        plt.show()
        plt.close('all')  # cegah penumpukan figure di memori selama loop 9 kombinasi

        # --- Checkpoint: simpan progres setelah tiap kombinasi selesai ---
        # Kalau sesi crash di tengah jalan, hasil kombinasi yang sudah selesai tidak hilang,
        # dan sel ini bisa dijalankan ulang -> otomatis lanjut (resume) dari sini.
        pd.DataFrame(walkforward_results).to_csv(_partial_path, index=False)
        pd.DataFrame(series_weights).to_csv(_weights_partial_path, index=False)
        print(f"  [checkpoint] progres tersimpan setelah {cat}-{state} "
              f"({len(walkforward_results)} baris hasil sejauh ini)")

walkforward_results_df = pd.DataFrame(walkforward_results)
walkforward_results_df.to_csv('walkforward_cnn_rnn_lstm_gru_dnn_arima_results.csv', index=False)

series_weights_df = pd.DataFrame(series_weights)
series_weights_df['Weight'] = series_weights_df['Weight_raw'] / series_weights_df['Weight_raw'].sum()
series_weights_df.to_csv('series_weights_wrmsse.csv', index=False)

print("\nHasil lengkap disimpan ke: walkforward_cnn_rnn_lstm_gru_dnn_arima_results.csv")
print("Bobot tiap deret (untuk WRMSSE) disimpan ke: series_weights_wrmsse.csv")
display(walkforward_results_df)
display(series_weights_df)

## 14. Tabel Perbandingan Akhir

Ringkasan performa 1D-CNN vs RNN vs LSTM vs GRU vs DNN vs ARIMA per metrik, dipecah menurut *State* x *Kategori*, ditutup dengan:
- **rata-rata sederhana** (unweighted, semua deret dibobot sama), dan
- **WRMSSE** — rata-rata RMSSE yang dibobot berdasarkan kontribusi revenue tiap deret (`series_weights_df`),
  mengikuti semangat skema bobot resmi kompetisi M5, supaya deret dengan revenue lebih besar berkontribusi
  lebih besar terhadap skor akhir.

In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

MODELS_ORDER = ['1D-CNN', 'RNN', 'LSTM', 'GRU', 'DNN', 'ARIMA']

for metric in ['RMSSE', 'MAE', 'RMSE', 'MAPE']:
    pivot = walkforward_results_df.pivot_table(
        index='State', columns=['Model', 'Category'], values=metric
    )
    pivot = pivot.reindex(columns=pd.MultiIndex.from_product(
        [MODELS_ORDER, CATEGORIES]
    ))
    print(f"\n=== Tabel {metric} -- 1D-CNN vs RNN vs LSTM vs GRU vs DNN vs ARIMA (Walk-Forward, Adil) ===")
    display(pivot)

# --- Ringkasan rata-rata SEDERHANA (unweighted, semua deret dibobot sama) ---
summary_unweighted = walkforward_results_df.groupby('Model')[['MAE', 'RMSE', 'MAPE', 'RMSSE']].mean().round(4)
summary_unweighted = summary_unweighted.reindex(MODELS_ORDER)
print("\n=== Ringkasan Rata-Rata SEDERHANA (unweighted, semakin kecil semakin baik) ===")
display(summary_unweighted)

# --- WRMSSE resmi ala M5: RMSSE per deret dibobot kontribusi revenue deret tsb ---
merged = walkforward_results_df.merge(
    series_weights_df[['Category', 'State', 'Weight']], on=['Category', 'State']
)
merged['Weighted_RMSSE'] = merged['RMSSE'] * merged['Weight']
wrmsse_summary = merged.groupby('Model')['Weighted_RMSSE'].sum().round(4).reindex(MODELS_ORDER)
wrmsse_summary = wrmsse_summary.to_frame(name='WRMSSE')
print("\n=== WRMSSE (Weighted RMSSE, dibobot kontribusi revenue ala M5) per Model ===")
display(wrmsse_summary)


## 15. Visualisasi Ringkasan Perbandingan Antar Model

Selain plot per deret (Bagian 13), bagian ini merangkum performa keenam model secara visual dalam satu
tempat:
- **Bar chart WRMSSE** per model (metrik ringkasan utama, dibobot kontribusi revenue).
- **Bar chart rata-rata RMSSE** unweighted per model.
- **Bar chart RMSSE per model, dipecah per kategori**, untuk melihat konsistensi performa lintas kategori.
- **Boxplot sebaran RMSSE** tiap model di seluruh deret (state x kategori), untuk melihat variabilitas performa,
  bukan cuma rata-ratanya.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- (a) Bar chart WRMSSE per model ---
wrmsse_plot = wrmsse_summary.reindex(MODELS_ORDER)['WRMSSE']
bars = axes[0].bar(wrmsse_plot.index, wrmsse_plot.values, color='tab:blue')
axes[0].set_title('WRMSSE per Model (semakin kecil semakin baik)')
axes[0].set_ylabel('WRMSSE')
axes[0].bar_label(bars, fmt='%.3f')

# --- (b) Bar chart rata-rata RMSSE unweighted per model ---
rmsse_plot = summary_unweighted.reindex(MODELS_ORDER)['RMSSE']
bars2 = axes[1].bar(rmsse_plot.index, rmsse_plot.values, color='tab:orange')
axes[1].set_title('Rata-Rata RMSSE Unweighted per Model')
axes[1].set_ylabel('RMSSE (rata-rata sederhana)')
axes[1].bar_label(bars2, fmt='%.3f')

plt.tight_layout()
plt.show()

# --- (c) Bar chart RMSSE per model, dipecah per kategori ---
cat_pivot = walkforward_results_df.pivot_table(
    index='Category', columns='Model', values='RMSSE', aggfunc='mean'
).reindex(columns=MODELS_ORDER)

ax = cat_pivot.plot(kind='bar', figsize=(10, 5))
ax.set_title('Rata-Rata RMSSE per Model, per Kategori')
ax.set_ylabel('RMSSE')
ax.set_xlabel('Kategori')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# --- (d) Boxplot sebaran RMSSE tiap model di seluruh deret (state x kategori) ---
box_data = [walkforward_results_df.loc[walkforward_results_df['Model'] == m, 'RMSSE'].values
            for m in MODELS_ORDER]

plt.figure(figsize=(9, 5))
plt.boxplot(box_data, labels=MODELS_ORDER, showmeans=True)
plt.title('Sebaran RMSSE per Model di Seluruh Deret (State x Kategori)')
plt.ylabel('RMSSE')
plt.xlabel('Model')
plt.tight_layout()
plt.show()

## 16. Kesimpulan

Bandingkan baris **WRMSSE** di atas terlebih dahulu — ini adalah metrik ringkasan utama, karena deret
dengan kontribusi revenue lebih besar diberi bobot lebih besar (mengikuti semangat skema penilaian resmi M5),
sehingga lebih adil dibanding rata-rata sederhana yang menganggap semua deret sama pentingnya.

Model dengan **WRMSSE** (dan `MAE` / `RMSE` / `MAPE` sebagai pelengkap) yang lebih kecil adalah model yang
lebih baik pada protokol walk-forward ini, di antara keenam model: **1D-CNN**, **RNN**, **LSTM**, **GRU**,
**DNN**, dan **ARIMA**. Perhatikan juga tabel per *state* x *kategori* untuk melihat apakah ada model yang
konsisten unggul di kombinasi tertentu (mis. model sequential seperti LSTM/GRU cenderung lebih kuat pada
deret dengan pola musiman/lag yang kompleks, sedangkan DNN sebagai baseline dense murni berguna untuk melihat
seberapa besar kontribusi struktur sekuensial terhadap akurasi dibanding sekadar melihat jendela fitur secara
flat).

In [ ]:
best_model_wrmsse = wrmsse_summary['WRMSSE'].idxmin()
best_model_unweighted = summary_unweighted['RMSSE'].idxmin()

print("=== Ringkasan Otomatis ===")
print(f"Model terbaik berdasarkan WRMSSE (metrik utama, dibobot revenue): "
      f"{best_model_wrmsse} (WRMSSE={wrmsse_summary.loc[best_model_wrmsse, 'WRMSSE']:.4f})")
print(f"Model terbaik berdasarkan rata-rata RMSSE unweighted (semua deret dibobot sama): "
      f"{best_model_unweighted} (RMSSE={summary_unweighted.loc[best_model_unweighted, 'RMSSE']:.4f})")

print("\nUrutan performa model berdasarkan WRMSSE (dari terbaik ke terburuk):")
display(wrmsse_summary.sort_values('WRMSSE'))

print("\nUrutan performa model berdasarkan rata-rata RMSSE unweighted (dari terbaik ke terburuk):")
display(summary_unweighted.sort_values('RMSSE')[['RMSSE']])

if best_model_wrmsse == best_model_unweighted:
    print(f"\nKedua metrik ringkasan (WRMSSE dan RMSSE unweighted) sepakat: model **{best_model_wrmsse}** "
          f"adalah yang paling akurat pada protokol walk-forward ini.")
else:
    print(f"\nCatatan: kedua metrik ringkasan tidak sepakat -> WRMSSE memilih {best_model_wrmsse}, "
          f"sedangkan rata-rata unweighted memilih {best_model_unweighted}. Ini mengindikasikan performa "
          f"{best_model_wrmsse} relatif lebih kuat pada deret ber-revenue besar, sementara "
          f"{best_model_unweighted} lebih konsisten baik di semua deret tanpa memandang skala revenue-nya.")